# Meeskond Operatsioonid - Week 8 API pipeline UrbanStyle.ltd jaoks

**Nädal 8: Python API ja automatiseeritud pipeline**  
**Meeskonnatöö: API päringud, puhastamine, RFM + marketing analytics, eksport ja automatiseerimine**

Notebook koondab Week 8 töö samas stiilis nagu Week 7 RFM notebook: iga roll on eraldi plokina, kood on käivitatav ning lõpus on Markole mõeldud äriline kokkuvõte. Week 8 edasiarendus muudab Week 7 analüüsi modulaarseks pipeline'iks, mis oskab andmeid API kaudu küsida, tulemusi valideerida ja failidesse eksportida.

**Rollid**
- Roll A: Data Fetching - Supabase API, pagination, retry ja fallback-andmed.
- Roll B: Transform - puhastamine, ühendamine, KPI-d, RFM, top kliendid ja kampaaniaplaan.
- Roll C: Visualization + Saving - juhendi järgi KPI kokkuvõte, nädalase tulu graafik ning CSV/HTML eksport.
- Roll D: Automation Script - kogu protsessi orkestreerimine, valideerimine ja failidesse eksport.

## Roll A: Data Fetching - API päringud ja fallback

Roll A eesmärk on tuua müügi-, kliendi- ja tooteandmed Supabase API-st. Pipeline sisaldab pagination'i, retry loogikat ning fallback'i, et analüüs oleks käivitatav ka siis, kui API võtmeid pole või ühendus ebaõnnestub.



In [57]:
# 1. TEEGID
# Impordime põhiteegid failiteede, ajatemplite ja tabelitöötluse jaoks.
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Kuvaseaded: näitame notebookis rohkem veerge ja hoiame tabelid loetavamad.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

# 2. PROJEKTI JA MOODULITE ASUKOHT
# Leiame projekti juurkausta, et notebook töötaks nii repo juurest kui week-08/team kaustast käivitades.
ROOT = Path.cwd()
if not (ROOT / 'week-08').exists():
    ROOT = Path.cwd().parents[1]

# Week 8 tiimitöö .py moodulid asuvad selles kaustas.
TEAM_DIR = ROOT / 'week-08' / 'team'

# Lisame tiimikausta import path'i, et Python leiaks data_fetcher.py, pipeline.py, transform.py ja visualize_export.py.
sys.path.insert(0, str(TEAM_DIR))

# 3. WEEK 8 MOODULITE IMPORT
# data_fetcher: Supabase API päringud ning fallback/sample andmed.
from data_fetcher import (
    create_supabase_client,
    csv_fallback_data,
    fetch_customers,
    fetch_products,
    fetch_sales,
    sample_data,
)

# pipeline: config laadimine ja analüüsi kuupäeva rakendamine.
from pipeline import apply_cli_date, load_config

# 4. CONFIG JA LIVE API VALIK
# Laeme config.yaml seaded ning määrame analüüsi lõppkuupäevaks 2025-02-28.
config = apply_cli_date(load_config(), '2025-02-28')

# True = notebook proovib esmalt Supabase API-st pärida.
# Kui ühendus ebaõnnestub, saab allpool kasutada fallback/sample andmeid.
USE_LIVE_API = True

# 5. ANDMETE LAADIMINE
# Kui live API on lubatud, loome Supabase kliendi ja pärime sales, customers ning products tabelid eraldi.
if USE_LIVE_API:
    try:
        supabase = create_supabase_client()
        date_filter = config['date_filter']
        pipeline_config = config['pipeline']
        table_config = config.get('tables', {})

        sales = fetch_sales(
            supabase,
            start_date=date_filter.get('start_date'),
            end_date=date_filter.get('end_date'),
            page_size=pipeline_config.get('page_size', 1000),
            max_retries=pipeline_config.get('max_retries', 3),
            retry_base_seconds=pipeline_config.get('retry_base_seconds', 1),
            table_name=table_config.get('sales', 'sales'),
        )

        customers = fetch_customers(
            supabase,
            page_size=pipeline_config.get('page_size', 1000),
            max_retries=pipeline_config.get('max_retries', 3),
            retry_base_seconds=pipeline_config.get('retry_base_seconds', 1),
            table_name=table_config.get('customers', 'customers'),
        )

        products = fetch_products(
            supabase,
            page_size=pipeline_config.get('page_size', 1000),
            max_retries=pipeline_config.get('max_retries', 3),
            retry_base_seconds=pipeline_config.get('retry_base_seconds', 1),
            table_name=table_config.get('products', 'products'),
        )

        data_source = 'supabase_api'
    except Exception as exc:
        print('Live API päring ebaõnnestus, kasutan fallback/sample andmeid:', exc)
        fallback = csv_fallback_data()
        if fallback is not None:
            sales, customers, products = fallback
            data_source = 'csv_fallback'
        else:
            sales, customers, products = sample_data()
            data_source = 'sample_data'
else:
    # Kui live API ei ole lubatud, proovime esmalt kohalikke CSV fallback-andmeid.
    fallback = csv_fallback_data()
    if fallback is not None:
        sales, customers, products = fallback
        data_source = 'csv_fallback'
    else:
        # Viimase varuna kasutame väikest näidisandmestikku, et notebook jääks käivitatavaks.
        sales, customers, products = sample_data()
        data_source = 'sample_data'

# 6. ESIMENE KONTROLL
# Prindime ainult laaditud tabelite suurused. Näidistabelit siin ei kuva.
print('Andmeallikas:', data_source)
print('Sales shape:', sales.shape)
print('Customers shape:', customers.shape)
print('Products shape:', products.shape)

Andmeallikas: supabase_api
Sales shape: (10086, 12)
Customers shape: (3150, 9)
Products shape: (362, 9)


## Roll B: Transform - puhastamine, KPI-d ja RFM

Roll B ühendab müügi-, kliendi- ja tooteandmed, eemaldab vigased read ning arvutab juhendis nõutud nädalased koondnäitajad ja KPI-d. Lisaks on meeskonna töös säilinud RFM segmentide ja kampaaniaplaani tabelid, sest need toetavad Markole antavat ärilist soovitust.

In [58]:
# 1. ANDMETE ÜHENDAMINE
# Liidame müügi- ja kliendiandmed customer_id põhjal.
merged = sales.merge(customers, on='customer_id', how='left', suffixes=('_sale', ''))

# Kui olemas on product_id, liidame juurde ka tooteandmed.
if not products.empty and 'product_id' in merged.columns and 'product_id' in products.columns:
    merged = merged.merge(products, on='product_id', how='left', suffixes=('', '_product'))

# Kui city veerg puudub või on tühi, kasutame vajadusel store_location/channel infot.
if 'city' not in merged.columns:
    merged['city'] = pd.NA
if 'store_location' in merged.columns:
    merged['city'] = merged['city'].fillna(merged['store_location'])
if 'channel' in merged.columns:
    online_mask = merged['city'].isna() & (merged['channel'].astype(str).str.lower() == 'online')
    merged.loc[online_mask, 'city'] = 'Online'
merged['city'] = merged['city'].fillna('Teadmata')

# 2. ANDMETE PUHASTAMINE
# Kontrollime, et edasiseks analüüsiks vajalikud veerud on olemas.
required_columns = {'customer_id', 'sale_date', 'total_price'}
missing_columns = sorted(required_columns - set(merged.columns))
if missing_columns:
    raise ValueError(f'Puuduvad kohustuslikud veerud: {missing_columns}')

# Eemaldame täielikud duplikaatread ja teisendame veerud õigesse tüüpi.
clean = merged.drop_duplicates().copy()
clean['sale_date'] = pd.to_datetime(clean['sale_date'], errors='coerce')
clean['customer_id'] = pd.to_numeric(clean['customer_id'], errors='coerce')
clean['total_price'] = pd.to_numeric(clean['total_price'], errors='coerce')

# Eemaldame read, kus puudub klient, kuupäev või müügisumma.
clean = clean.dropna(subset=['customer_id', 'sale_date', 'total_price']).copy()

# Alles jäävad ainult positiivse müügisummaga read.
clean = clean[clean['total_price'] > 0].copy()
clean['customer_id'] = clean['customer_id'].astype(int)

# Kui sale_id puudub, loome selle, et tellimuste arvu oleks võimalik lugeda.
if 'sale_id' not in clean.columns:
    clean['sale_id'] = range(1, len(clean) + 1)

# Lisame puuduvad kliendiinfo veerud, et RFM ja kontaktitavuse loogika töötaks ühtemoodi.
for column in ['email', 'phone', 'first_name', 'last_name', 'city', 'loyalty_tier']:
    if column not in clean.columns:
        clean[column] = pd.NA

# Loome abiveerud, mis näitavad, kas kliendil on email, telefon või vähemalt üks kontaktkanal.
clean['has_email'] = clean['email'].notna() & (clean['email'].astype(str).str.strip() != '')
clean['has_phone'] = clean['phone'].notna() & (clean['phone'].astype(str).str.strip() != '')
clean['has_contact'] = clean['has_email'] | clean['has_phone']

# 3. KONTROLL
# Kuvame tabelite mõõdud ja puhastatud andmete esimesed read.
print('Ühendatud shape:', merged.shape)
print('Puhastatud shape:', clean.shape)
clean.head()

Ühendatud shape: (10086, 28)
Puhastatud shape: (8866, 31)


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year,product_name,category,subcategory,supplier,cost_price,retail_price,eco_certified,created_at,has_email,has_phone,has_contact
0,1,1,INV-202301-00001,2023-01-10,2588,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,Paju,None,+37254290294,Tallinn,2022-07-28,bronze,1997.0,Soe teksane chino püksid,Meeste_riided,püksid,Tartu Tekstiil AS,156.71,234.79,False,2024-09-13,False,True,True
1,2,2,INV-202301-00002,2023-01-16,4338,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,Luik,merle.luik@mail.ee,+37251501812,Tallinn,2020-09-22,None,1996.0,Õhuline seemisnahkne rannasandaalid,Jalanõusid,sandaalid,Türgi Tekstil A.Ş.,170.10,241.13,True,2021-10-18,True,True,True
2,3,3,INV-202301-00003,2023-01-05,4673,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,Saar,liina.saar@gmail.com,+37288097990,Tallinn,2020-03-31,silver,1973.0,Mugav goretex platvormi kontsad,Jalanõusid,kontsad,Itaalia Moda SRL,141.90,221.19,False,2024-11-21,True,True,True
3,4,4,INV-202301-00004,2023-01-02,4677,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,Pihl,aili.pihl@yahoo.com,+37283754888,Narva,2021-10-08,gold,1972.0,Stiilne siidine labakindad,Aksessuaarid,kindad,Leedu Linane UAB,27.64,45.21,False,2020-07-06,True,True,True
4,5,5,INV-202301-00005,2023-01-13,2390,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,Lill,triin.lill@telia.ee,+37253780596,Tartu,2021-04-09,None,1996.0,Vintage softshell alussärk,Laste_riided,trikotaaž,Vilma Design OÜ,59.84,99.57,True,2022-11-22,True,True,True


In [59]:
# 1. ANALÜÜSIPERIOOD
# Kasutame config failist tulevat reference_date väärtust ja jätame alles ainult selle kuupäevani toimunud müügid.
reference_date = config['pipeline'].get('reference_date')
analysis_data = clean.copy()
if reference_date:
    cutoff = pd.to_datetime(reference_date)
    analysis_data = analysis_data[analysis_data['sale_date'] <= cutoff].copy()

# 2. NÄDALASED KOONDNÄITAJAD
# Grupeerime müügid nädala kaupa ning arvutame tulu, tellimuste arvu ja unikaalsete klientide arvu.
weekly = (
    analysis_data.resample('W-MON', on='sale_date', label='left', closed='left')
    .agg(
        revenue=('total_price', 'sum'),
        orders=('sale_id', 'count'),
        unique_customers=('customer_id', 'nunique'),
    )
    .reset_index()
    .rename(columns={'sale_date': 'week'})
)

# Lisame nädalatele aasta, nädala numbri, loetava sildi ja keskmise ostukorvi.
iso_calendar = weekly['week'].dt.isocalendar()
weekly['week_year'] = iso_calendar['year'].astype(int)
weekly['week_number'] = iso_calendar['week'].astype(int)
weekly['week_label'] = weekly['week_year'].astype(str) + ' nädal ' + weekly['week_number'].astype(str)
weekly['avg_order_value'] = weekly['revenue'] / weekly['orders']
weekly[['revenue', 'avg_order_value']] = weekly[['revenue', 'avg_order_value']].round(2)

# 3. KPI-D JA ANDMEKVALITEET
# KPI-d arvutame otse puhastatud müügitabelilt.
orders = len(analysis_data)
total_revenue = float(analysis_data['total_price'].sum())
unique_customers = int(analysis_data['customer_id'].nunique())
kpis = {
    'total_revenue': round(total_revenue, 2),
    'orders': int(orders),
    'unique_customers': unique_customers,
    'avg_order_value': round(total_revenue / orders, 2) if orders else 0.0,
    'revenue_per_customer': round(total_revenue / unique_customers, 2) if unique_customers else 0.0,
}

# Andmekvaliteedi raport võrdleb algset ühendatud tabelit ja puhastatud analüüsitabelit.
raw_total_price = pd.to_numeric(merged['total_price'], errors='coerce') if 'total_price' in merged.columns else pd.Series(dtype=float)
raw_sale_date = pd.to_datetime(merged['sale_date'], errors='coerce') if 'sale_date' in merged.columns else pd.Series(dtype='datetime64[ns]')
data_quality = pd.DataFrame(
    [
        ('rows_before_cleaning', len(merged), 'Sisendridade arv enne puhastamist'),
        ('rows_after_cleaning', len(analysis_data), 'Ridade arv pärast puhastamist'),
        ('rows_removed', len(merged) - len(analysis_data), 'Puhastamisel eemaldatud read'),
        ('duplicate_rows', int(merged.duplicated().sum()) if not merged.empty else 0, 'Täielikud duplikaatread'),
        ('missing_customer_id', int(merged['customer_id'].isna().sum()) if 'customer_id' in merged.columns else 0, 'Puuduva customer_id-ga read'),
        ('invalid_sale_date', int(raw_sale_date.isna().sum()), 'Puuduvad või vigased kuupäevad'),
        ('non_positive_revenue', int((raw_total_price <= 0).sum()), 'Nullist väiksema või võrdse summaga read'),
        ('missing_contact_clean', int((~analysis_data['has_contact']).sum()) if 'has_contact' in analysis_data.columns else 0, 'Puhastes andmetes kontaktita read'),
    ],
    columns=['metric', 'value', 'description'],
)

# 4. RFM ARVUTUS
# Koondame iga kliendi kohta esimese/viimase ostu, sageduse ja kogukulutuse.
rfm_reference_date = pd.to_datetime(reference_date) if reference_date else analysis_data['sale_date'].max() + pd.Timedelta(days=1)
rfm = (
    analysis_data.groupby('customer_id')
    .agg(
        first_purchase_date=('sale_date', 'min'),
        last_purchase_date=('sale_date', 'max'),
        frequency=('sale_id', 'count'),
        monetary_value=('total_price', 'sum'),
        first_name=('first_name', 'first'),
        last_name=('last_name', 'first'),
        email=('email', 'first'),
        phone=('phone', 'first'),
        has_email=('has_email', 'max'),
        has_phone=('has_phone', 'max'),
        has_contact=('has_contact', 'max'),
        city=('city', 'first'),
        loyalty_tier=('loyalty_tier', 'first'),
    )
    .reset_index()
)

rfm['recency_days'] = (rfm_reference_date - rfm['last_purchase_date']).dt.days
rfm['customer_lifespan_days'] = (rfm['last_purchase_date'] - rfm['first_purchase_date']).dt.days.clip(lower=0)
rfm['avg_order_value'] = rfm['monetary_value'] / rfm['frequency']

# R_score: väiksem recency on parem, seega skoorid lähevad 5-st 1-ni.
if rfm['recency_days'].nunique() < 2:
    rfm['R_score'] = 5
else:
    rfm['R_score'] = pd.qcut(rfm['recency_days'].rank(method='first'), min(5, rfm['recency_days'].nunique()), labels=[5, 4, 3, 2, 1][:min(5, rfm['recency_days'].nunique())]).astype(int)

# F_score ja M_score: suurem sagedus ning suurem kogukulutus on paremad.
if rfm['frequency'].nunique() < 2:
    rfm['F_score'] = 5
else:
    rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), min(5, rfm['frequency'].nunique()), labels=[1, 2, 3, 4, 5][:min(5, rfm['frequency'].nunique())]).astype(int)

if rfm['monetary_value'].nunique() < 2:
    rfm['M_score'] = 5
else:
    rfm['M_score'] = pd.qcut(rfm['monetary_value'].rank(method='first'), min(5, rfm['monetary_value'].nunique()), labels=[1, 2, 3, 4, 5][:min(5, rfm['monetary_value'].nunique())]).astype(int)

# Summeerime RFM skoorid ja määrame segmendid.
rfm['RFM_Score'] = rfm[['R_score', 'F_score', 'M_score']].sum(axis=1)
rfm['Segment'] = pd.cut(
    rfm['RFM_Score'],
    bins=[-1, 3, 6, 9, 12, 15],
    labels=['Lost', 'At Risk', 'Potential', 'Loyal', 'VIP Champions'],
)

# Loome kliendi kuvatava nime ja ümardame rahalised näitajad.
rfm['customer_name'] = (rfm['first_name'].fillna('').astype(str).str.strip() + ' ' + rfm['last_name'].fillna('').astype(str).str.strip()).str.strip()
rfm.loc[rfm['customer_name'] == '', 'customer_name'] = 'Klient ' + rfm['customer_id'].astype(str)
rfm[['monetary_value', 'avg_order_value']] = rfm[['monetary_value', 'avg_order_value']].round(2)
rfm = rfm.sort_values(['RFM_Score', 'monetary_value'], ascending=False)

# 5. SEGMENTIDE KOKKUVÕTE
# Segmentide kokkuvõte näitab iga segmendi suurust, käivet ja kontaktitavust.
segment_summary = (
    rfm.groupby('Segment', observed=False)
    .agg(
        customers=('customer_id', 'count'),
        avg_recency_days=('recency_days', 'mean'),
        avg_frequency=('frequency', 'mean'),
        total_revenue=('monetary_value', 'sum'),
        avg_monetary_value=('monetary_value', 'mean'),
        reachable_customers=('has_contact', 'sum'),
    )
    .reset_index()
)
segment_summary = segment_summary[segment_summary['customers'] > 0].copy()
segment_summary['customer_share_pct'] = segment_summary['customers'] / segment_summary['customers'].sum() * 100
segment_summary['revenue_share_pct'] = segment_summary['total_revenue'] / segment_summary['total_revenue'].sum() * 100
segment_summary['reachable_share_pct'] = segment_summary['reachable_customers'] / segment_summary['customers'] * 100
segment_summary = segment_summary.sort_values('total_revenue', ascending=False).round(2)

# 6. KAMPAANIAPLAAN
# Koostame igale RFM segmendile tegevusplaani.
campaign_plan = pd.DataFrame(
    [
        ('VIP Champions', 'Hoida parimaid kliente', 'Early access ja personaalsed VIP pakkumised', 'Email + SMS', 'VIP preview / tasuta tarne', 'Repeat purchase rate'),
        ('Loyal', 'Kasvatada ostukorvi', 'Cross-sell ja komplektpakkumised', 'Email', 'Bundle offer', 'Avg order value'),
        ('Potential', 'Muuta teine ost harjumuseks', 'Järgmise ostu stiimul', 'Email', '-10% järgmisele ostule', 'Second purchase conversion'),
        ('At Risk', 'Võita klient tagasi enne kadumist', 'Piiratud ajaga win-back sõnum', 'Email + SMS', 'Isiklik sooduskood', 'Reactivation rate'),
        ('Lost', 'Proovida odavat taasaktiveerimist', 'Viimane tagasituleku pakkumine', 'Email', 'Tugevam ühekordne soodustus', 'Campaign ROI'),
    ],
    columns=['Segment', 'goal', 'message', 'channel', 'offer', 'primary_metric'],
)
campaign_plan = campaign_plan.merge(segment_summary[['Segment', 'customers', 'total_revenue', 'revenue_share_pct']], on='Segment', how='left')

# 7. ÄRILINE TÕLGENDUS
# Koostame Markole lühikese ärilise kokkuvõtte arvutatud RFM tulemuste põhjal.
total_customers = int(rfm['customer_id'].nunique())
rfm_total_revenue = rfm['monetary_value'].sum()
vip_customers = int((rfm['Segment'] == 'VIP Champions').sum())
at_risk_customers = int((rfm['Segment'] == 'At Risk').sum())
lost_customers = int((rfm['Segment'] == 'Lost').sum())
vip_revenue = rfm.loc[rfm['Segment'] == 'VIP Champions', 'monetary_value'].sum()
at_risk_revenue = rfm.loc[rfm['Segment'] == 'At Risk', 'monetary_value'].sum()
vip_share = vip_revenue / rfm_total_revenue * 100 if rfm_total_revenue else 0
at_risk_share = at_risk_revenue / rfm_total_revenue * 100 if rfm_total_revenue else 0

business_interpretation = f'''
Week 8 API RFM raport

UrbanStyle andmestikus on {total_customers} analüüsitavat klienti, kellest {vip_customers} kuuluvad VIP Champions segmenti.
VIP kliendid annavad {vip_share:.1f}% kogukäibest.
At Risk segmendis on {at_risk_customers} klienti ja Lost segmendis {lost_customers} klienti.
At Risk segment annab veel {at_risk_share:.1f}% käibest.

Soovitused Markole:
1. VIP Champions: käivita early access programm, personaalsed pakkumised ja VIP sooduskoodid.
2. At Risk: saada win-back pakkumine enne, kui kliendid liiguvad Lost segmenti.
3. Potential ja Loyal: kasvata neid lojaalsusprogrammi ja cross-sell pakkumistega VIP segmendiks.
'''.strip()

# 5. RESULTS SÕNASTIK
# Koondame kõik vahetulemused ühte sõnastikku, et järgmised lahtrid saaksid samu võtmeid kasutada.
results = {
    'data_source': data_source,
    'clean_sales': analysis_data,
    'data_quality': data_quality,
    'weekly': weekly,
    'kpis': kpis,
    'rfm': rfm,
    'segment_summary': segment_summary,
    'campaign_plan': campaign_plan,
    'business_interpretation': business_interpretation,
}

# 6. KPI-D
# Kuvame Markole juhtimiseks kõige olulisemad koondnäitajad.
print('KPI-d:')
display(pd.DataFrame([results['kpis']]))

# 7. ANDMEKVALITEET
# Kuvame, mitu rida puhastuse käigus eemaldati ja miks.
print('Andmekvaliteet:')
display(results['data_quality'])

# 8. RFM SEGMENTIDE KOKKUVÕTE
# Kuvame klientide arvu, käibe ja kontaktitavuse segmendi kaupa.
print('RFM segmentide kokkuvõte:')
display(results['segment_summary'])

KPI-d:


,total_revenue,orders,unique_customers,avg_order_value,revenue_per_customer
0,2655584.91,8866,2538,299.52,1046.33


Andmekvaliteet:


,metric,value,description
0,rows_before_cleaning,10086,Sisendridade arv enne puhastamist
1,rows_after_cleaning,8866,Ridade arv pärast puhastamist
2,rows_removed,1220,Puhastamisel eemaldatud read
3,duplicate_rows,60,Täielikud duplikaatread
4,missing_customer_id,987,Puuduva customer_id-ga read
5,invalid_sale_date,0,Puuduvad või vigased kuupäevad
6,non_positive_revenue,197,Nullist väiksema või võrdse summaga read
7,missing_contact_clean,0,Puhastes andmetes kontaktita read


RFM segmentide kokkuvõte:


,Segment,customers,avg_recency_days,avg_frequency,total_revenue,avg_monetary_value,reachable_customers,customer_share_pct,revenue_share_pct,reachable_share_pct
4,VIP Champions,454,64.76,7.63,1138229.81,2507.11,454,17.89,42.86,100.0
3,Loyal,669,150.11,3.80,779244.04,1164.79,669,26.36,29.34,100.0
2,Potential,773,210.56,2.48,528841.19,684.14,773,30.46,19.91,100.0
1,At Risk,527,312.10,1.57,189975.82,360.49,527,20.76,7.15,100.0
0,Lost,115,518.90,1.00,19294.05,167.77,115,4.53,0.73,100.0


## Roll C: Visualization + Saving - diagrammid ja ekspordid

Roll C loob juhendis nõutud Plotly väljundid: nädalase tulu joondiagrammi ja KPI kokkuvõtte. Lisaks loome RFM segmentide ja Top 10 klientide visuaalid ning salvestame CSV/HTML failid ajatempliga failinimedega.

In [60]:
# 1. KPI KOKKUVÕTE
# Võtame KPI dict'i väärtused eraldi muutujasse, et tabeli koostamine oleks notebookis nähtav.
kpis = results['kpis']

# Paneme KPI nimetused ja väärtused kahte nimekirja, sest Plotly Table ootab veergude kaupa andmeid.
kpi_labels = [
    'Total revenue',
    'Orders',
    'Unique customers',
    'Avg order value',
    'Revenue per customer',
]

kpi_values = [
    f"{kpis['total_revenue']:.2f} EUR",
    kpis['orders'],
    kpis['unique_customers'],
    f"{kpis['avg_order_value']:.2f} EUR",
    f"{kpis['revenue_per_customer']:.2f} EUR",
]

# Loome Plotly tabeli, kus vasakul on KPI nimi ja paremal arvutatud väärtus.
kpi_fig = go.Figure(
    data=[
        go.Table(
            header={'values': ['KPI', 'Väärtus'], 'fill_color': '#d9ead3', 'align': 'left'},
            cells={'values': [kpi_labels, kpi_values], 'align': 'left'},
        )
    ]
)

kpi_fig.show()

# 2. NÄDALASE TULU GRAAFIK
# Võtame nädalased koondnäitajad eraldi DataFrame'i, et graafiku sisend oleks selgelt näha.
weekly = results['weekly']

# Loome joondiagrammi: x-teljel on nädal, y-teljel nädala kogutulu.
weekly_fig = px.line(
    weekly,
    x='week',
    y='revenue',
    markers=True,
    title='Nädalane tulu',
    labels={'week': 'Nädal', 'revenue': 'Tulu (EUR)'},
    custom_data=['week_label', 'orders', 'unique_customers'],
)

# Lisame hover-teksti, et graafikul oleks näha nädal, tulu, tellimused ja unikaalsed kliendid.
weekly_fig.update_traces(
    hovertemplate=(
        'Nädal=%{customdata[0]}<br>'
        'Tulu (EUR)=%{y:.2f}<br>'
        'Tellimusi=%{customdata[1]}<br>'
        'Unikaalseid kliente=%{customdata[2]}<extra></extra>'
    )
)

weekly_fig.show()

In [61]:
# 1. NÄDALASED KOONDNÄITAJAD
# Kuvame täiendavad tulemused tabelitena ja lisame RFM segmentide ning Top 10 klientide visuaalid.
print('Nädalased koondnäitajad:')
display(results['weekly'].head())

# 2. RFM SEGMENTIDE TABEL JA VISUAAL
# Segmentide tabel näitab, mitu klienti ja kui palju käivet on igas segmendis.
segment_summary = results['segment_summary'].copy()
print('RFM segmentide tabel:')
display(segment_summary)

rfm_segment_fig = px.bar(
    segment_summary.sort_values('customers', ascending=False),
    x='Segment',
    y='customers',
    color='Segment',
    text='customers',
    title='RFM segmentide jaotus klientide arvu järgi',
    labels={'Segment': 'Segment', 'customers': 'Klientide arv'},
)
rfm_segment_fig.update_layout(showlegend=False)
rfm_segment_fig.show()

# 3. TOP 10 KLIENTI TABELI JA VISUAALINA
# Võtame RFM tabelist 10 suurima kogukulutusega klienti ja kuvame need tabelina ning tulpdiagrammina.
top_10_clients = results['rfm'].nlargest(10, 'monetary_value')[
    ['customer_id', 'customer_name', 'Segment', 'frequency', 'monetary_value', 'avg_order_value']
].copy()

print('Top 10 klienti kogukulutuse järgi:')
display(top_10_clients)

top_10_fig = px.bar(
    top_10_clients.sort_values('monetary_value'),
    x='monetary_value',
    y='customer_name',
    color='Segment',
    orientation='h',
    text='monetary_value',
    title='Top 10 klienti kogukulutuse järgi',
    labels={'monetary_value': 'Kogukulutus (EUR)', 'customer_name': 'Klient'},
)
top_10_fig.update_traces(texttemplate='%{text:.0f} EUR', textposition='outside')
top_10_fig.show()


Nädalased koondnäitajad:


,week,revenue,orders,unique_customers,week_year,week_number,week_label,avg_order_value
0,2022-12-26,3424.32,9,9,2022,52,2022 nädal 52,380.48
1,2023-01-02,16908.79,64,61,2023,1,2023 nädal 1,264.20
2,2023-01-09,18062.69,51,50,2023,2,2023 nädal 2,354.17
3,2023-01-16,18994.68,52,50,2023,3,2023 nädal 3,365.28
4,2023-01-23,14058.45,46,46,2023,4,2023 nädal 4,305.62


RFM segmentide tabel:


,Segment,customers,avg_recency_days,avg_frequency,total_revenue,avg_monetary_value,reachable_customers,customer_share_pct,revenue_share_pct,reachable_share_pct
4,VIP Champions,454,64.76,7.63,1138229.81,2507.11,454,17.89,42.86,100.0
3,Loyal,669,150.11,3.80,779244.04,1164.79,669,26.36,29.34,100.0
2,Potential,773,210.56,2.48,528841.19,684.14,773,30.46,19.91,100.0
1,At Risk,527,312.10,1.57,189975.82,360.49,527,20.76,7.15,100.0
0,Lost,115,518.90,1.00,19294.05,167.77,115,4.53,0.73,100.0


Top 10 klienti kogukulutuse järgi:


,customer_id,customer_name,Segment,frequency,monetary_value,avg_order_value
1365,3618,Tiina Pärn,VIP Champions,72,27920.86,387.79
1134,3350,Priit Rand,VIP Champions,76,26286.10,345.87
750,2889,Laura Tammik,VIP Champions,70,23892.45,341.32
1391,3648,Erkki Ilves,VIP Champions,72,22942.42,318.64
837,2997,Kevin Org,VIP Champions,74,22933.85,309.92
1868,4206,Kersti Lill,VIP Champions,67,21526.69,321.29
1353,3605,Anu Kuusik,VIP Champions,76,21482.68,282.67
187,2221,Riina Lill,VIP Champions,64,21313.81,333.03
128,2154,Annika Saar,VIP Champions,65,20343.83,312.98
1456,3722,Ago Kull,VIP Champions,62,20317.81,327.71


In [62]:
# 1. KAMPAANIAPLAAN
# Seome iga RFM segmendi konkreetse sõnumi, kanali, pakkumise ja mõõdikuga.
print('Kampaaniaplaan:')
display(results['campaign_plan'])

Kampaaniaplaan:


,Segment,goal,message,channel,offer,primary_metric,customers,total_revenue,revenue_share_pct
0,VIP Champions,Hoida parimaid kliente,Early access ja personaalsed VIP pakkumised,Email + SMS,VIP preview / tasuta tarne,Repeat purchase rate,454,1138229.81,42.86
1,Loyal,Kasvatada ostukorvi,Cross-sell ja komplektpakkumised,Email,Bundle offer,Avg order value,669,779244.04,29.34
2,Potential,Muuta teine ost harjumuseks,Järgmise ostu stiimul,Email,-10% järgmisele ostule,Second purchase conversion,773,528841.19,19.91
3,At Risk,Võita klient tagasi enne kadumist,Piiratud ajaga win-back sõnum,Email + SMS,Isiklik sooduskood,Reactivation rate,527,189975.82,7.15
4,Lost,Proovida odavat taasaktiveerimist,Viimane tagasituleku pakkumine,Email,Tugevam ühekordne soodustus,Campaign ROI,115,19294.05,0.73


## Roll D: Automation Script - valideerimine ja eksport

Roll D seob rollid A-C üheks pipeline'iks: `extract -> transform -> validate -> export`. Notebookis käivitame valideerimise ja piiratud ekspordi eraldi, et oleks selgelt näha, millised Week 8 põhinõuete failid tekivad.

In [63]:
# 1. VALIDEERIMINE
# Kontrollime, et põhitabelid ei oleks tühjad ja käibesummad klapiksid eri raportite vahel.
validation_checks = {
    'clean_sales_not_empty': not results['clean_sales'].empty,
    'weekly_not_empty': not results['weekly'].empty,
    'rfm_not_empty': not results['rfm'].empty,
    'data_quality_not_empty': not results['data_quality'].empty,
    'campaign_plan_not_empty': not results['campaign_plan'].empty,
    'revenue_matches_rfm': round(results['clean_sales']['total_price'].sum(), 2) == round(results['rfm']['monetary_value'].sum(), 2),
    'revenue_matches_weekly': round(results['clean_sales']['total_price'].sum(), 2) == round(results['weekly']['revenue'].sum(), 2),
}

for check_name, check_ok in validation_checks.items():
    print(check_name, 'OK' if check_ok else 'PROBLEEM')

if not all(validation_checks.values()):
    raise RuntimeError('Pipeline valideerimine ebaõnnestus.')

# 2. VÄLJUNDKAUST JA AJATEMPEL
# Loome output kausta ja ajatembli, mida kasutatakse failinimedes.
output_dir = TEAM_DIR / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
date_str = datetime.now().strftime('%Y%m%d_%H%M%S')

# 3. CSV VÄLJUNDID
# Salvestame nädalased koondnäitajad ja KPI-d ajatempliga CSV failidesse.
weekly_csv = output_dir / f'weekly_aggregates_{date_str}.csv'
kpis_csv = output_dir / f'kpis_{date_str}.csv'
results['weekly'].to_csv(weekly_csv, index=False, encoding='utf-8')
pd.DataFrame([results['kpis']]).to_csv(kpis_csv, index=False, encoding='utf-8')

# 4. HTML VÄLJUNDID
# Salvestame juhendis nõutud nädalase tulujoonise ja KPI kokkuvõtte HTML failidena.
weekly_chart = output_dir / f'weekly_revenue_{date_str}.html'
kpi_chart = output_dir / f'kpi_summary_{date_str}.html'
rfm_segment_chart = output_dir / f'rfm_segmentide_jaotus_{date_str}.html'
top_10_chart = output_dir / f'rfm_top_10_kliendid_{date_str}.html'
weekly_fig.write_html(weekly_chart)
kpi_fig.write_html(kpi_chart)
rfm_segment_fig.write_html(rfm_segment_chart)
top_10_fig.write_html(top_10_chart)

# 5. VÄLJUNDITE KOONDNIMEKIRI
# Kogume loodud failiteed sõnastikku, et neid allpool mugavalt kuvada.
paths = {
    'weekly_csv': weekly_csv,
    'kpis_csv': kpis_csv,
    'weekly_chart': weekly_chart,
    'kpi_chart': kpi_chart,
    'rfm_segment_chart': rfm_segment_chart,
    'top_10_chart': top_10_chart,
}

# 6. EKSPORDI KONTROLL
# Prindime välja, mitu faili tekkis ja kuhu need salvestati.
print('Valideerimine õnnestus.')
print('Väljundkaust:', output_dir)
print('Failide arv:', len(paths))

# 7. LOODUD FAILID
# Kuvame iga loodud faili nime ja asukoha.
for key, path in paths.items():
    print(f'{key}: {path}')

clean_sales_not_empty OK
weekly_not_empty OK
rfm_not_empty OK
data_quality_not_empty OK
campaign_plan_not_empty OK
revenue_matches_rfm OK
revenue_matches_weekly OK
Valideerimine õnnestus.
Väljundkaust: c:\Users\Kätlin\Documents\Õppeprojekt\daca-portfolio\week-08\team\output
Failide arv: 6
weekly_csv: c:\Users\Kätlin\Documents\Õppeprojekt\daca-portfolio\week-08\team\output\weekly_aggregates_20260522_133512.csv
kpis_csv: c:\Users\Kätlin\Documents\Õppeprojekt\daca-portfolio\week-08\team\output\kpis_20260522_133512.csv
weekly_chart: c:\Users\Kätlin\Documents\Õppeprojekt\daca-portfolio\week-08\team\output\weekly_revenue_20260522_133512.html
kpi_chart: c:\Users\Kätlin\Documents\Õppeprojekt\daca-portfolio\week-08\team\output\kpi_summary_20260522_133512.html
rfm_segment_chart: c:\Users\Kätlin\Documents\Õppeprojekt\daca-portfolio\week-08\team\output\rfm_segmentide_jaotus_20260522_133512.html
top_10_chart: c:\Users\Kätlin\Documents\Õppeprojekt\daca-portfolio\week-08\team\output\rfm_top_10_kliend

## Äritõlgendus Markole

Allolev tekst koostatakse pipeline'i RFM tulemuste põhjal automaatselt. See on mõeldud juhtimisotsuse toetamiseks: millised segmendid vajavad hoidmist, millised taasaktiveerimist ning kuidas kampaaniate mõju mõõta.

In [64]:
# 1. ÄRILINE TÕLGENDUS
# Prindime automaatselt koostatud soovitused RFM tulemuste põhjal.
print(results['business_interpretation'])

# Notebook lõpeb ärilise tõlgendusega. Notificationi kokkuvõtet siin ei kuvata.

Week 8 API RFM raport

UrbanStyle andmestikus on 2538 analüüsitavat klienti, kellest 454 kuuluvad VIP Champions segmenti.
VIP kliendid annavad 42.9% kogukäibest.
At Risk segmendis on 527 klienti ja Lost segmendis 115 klienti.
At Risk segment annab veel 7.2% käibest.

Soovitused Markole:
1. VIP Champions: käivita early access programm, personaalsed pakkumised ja VIP sooduskoodid.
2. At Risk: saada win-back pakkumine enne, kui kliendid liiguvad Lost segmenti.
3. Potential ja Loyal: kasvata neid lojaalsusprogrammi ja cross-sell pakkumistega VIP segmendiks.


## Kokkuvõte: soovitused ja refleksioon

**Mis Week 8-s muutus võrreldes Week 7-ga?**  
Week 7 RFM analüüs oli notebookipõhine analüüs. Week 8 muudab sama loogika tootmislikumaks pipeline'iks: andmed tulevad API kaudu, vead logitakse, päringuid proovitakse uuesti, tulemused valideeritakse ning väljundid salvestatakse automaatselt.

**Mida Marko saab kasutada?**  
Kõige praktilisemad väljundid selles notebookis on ajatempliga `weekly_aggregates_*.csv`, `kpis_*.csv`, `weekly_revenue_*.html` ja `kpi_summary_*.html`. Töödeldud tabelid ja jagatav HTML visualisatsioon.

**Soovitus järgmise sammuna**  
Käivitada At Risk segmendile win-back kampaania ning mõõta tulemusi taasostu määra, käivet kliendi kohta ja kampaania ROI-d.

**AI kasutamine**  
AI aitas Week 7 RFM loogika muuta Week 8 modulaarseks API pipeline'iks, lisada retry, fallback, logimise, valideerimise, ekspordi, dashboardi ja marketingi tegevusplaani.